In [1]:
import pandas as pd
import numpy as np

In [6]:
import openpyxl

In [7]:
df = pd.read_excel('./data/NATCOR_corridor_panel_raw.xlsx')
print(f"Raw shape: {df.shape}")
print(f"Raw null count: {df.isnull().sum().sum()}")

Raw shape: (722, 14)
Raw null count: 18


In [9]:
# 1. Strip whitespace from Corridor 
df['Corridor'] = df['Corridor'].str.strip()
print(f"\n[1] Corridor names standardised. Unique: {sorted(df['Corridor'].unique())}")


[1] Corridor names standardised. Unique: ['CAPE', 'IRON_ORE', 'NATCOR', 'NE_CORRIDOR', 'NORTHCOR', 'SOUTHCOR']


In [10]:
# 2. Drop exact duplicate rows 
before = len(df)
df = df.drop_duplicates(subset=['Corridor', 'Date'], keep='first').copy()
print(f"\n[2] Dropped {before - len(df)} duplicate rows. Shape now: {df.shape}")


[2] Dropped 2 duplicate rows. Shape now: (720, 14)


In [11]:
# 3. Cap Loco_Avail_Rate at 1.0 
bad_loco = df['Loco_Avail_Rate'] > 1.0
print(f"\n[3] Loco_Avail_Rate > 1.0 found: {bad_loco.sum()} rows")
print(df.loc[bad_loco, ['Date', 'Corridor', 'Loco_Avail_Rate']])
df.loc[bad_loco, 'Loco_Avail_Rate'] = np.nan   # set to NaN -> will be interpolated below
print("    Set to NaN (will be interpolated).")


[3] Loco_Avail_Rate > 1.0 found: 1 rows
           Date     Corridor  Loco_Avail_Rate
413  2019-05-01  NE_CORRIDOR             1.85
    Set to NaN (will be interpolated).


In [12]:
# 4. Avg_Delay_Hrs extreme outlier 
bad_delay = df['Avg_Delay_Hrs'] > 200
print(f"\n[4] Avg_Delay_Hrs > 200 found: {bad_delay.sum()} rows")
print(df.loc[bad_delay, ['Date', 'Corridor', 'Avg_Delay_Hrs']])
df.loc[bad_delay, 'Avg_Delay_Hrs'] = np.nan
print("    Set to NaN (will be interpolated).")


[4] Avg_Delay_Hrs > 200 found: 1 rows
           Date  Corridor  Avg_Delay_Hrs
559  2021-07-01  NORTHCOR          987.0
    Set to NaN (will be interpolated).


In [13]:
# 5. Rail_Vol_mt == 0 (CAPE 2018-11)
zero_rail = (df['Rail_Vol_mt'] == 0)
print(f"\n[5] Rail_Vol_mt = 0 found: {zero_rail.sum()} rows")
print(df.loc[zero_rail, ['Date', 'Corridor', 'Rail_Vol_mt']])
df.loc[zero_rail, 'Rail_Vol_mt'] = np.nan
print("    Set to NaN (will be interpolated).")


[5] Rail_Vol_mt = 0 found: 1 rows
          Date Corridor  Rail_Vol_mt
46  2018-11-01     CAPE          0.0
    Set to NaN (will be interpolated).


In [14]:
#  6. Verify NATCOR 2023 Jan-Mar now present (they were the 'NATCOR ' rows) 
natcor_2023 = df[(df['Corridor'] == 'NATCOR') & (df['Year'] == 2023)]
months_2023 = sorted(natcor_2023['Month'].tolist())
print(f"\n[6] NATCOR 2023 months present after name-strip: {months_2023}")
assert set(range(1,13)).issubset(set(months_2023)), "NATCOR 2023 still has missing months!"
print("    All 12 months confirmed — the 'NATCOR ' (space) rows were the 2023 Jan-Mar data.")


[6] NATCOR 2023 months present after name-strip: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
    All 12 months confirmed — the 'NATCOR ' (space) rows were the 2023 Jan-Mar data.


In [15]:
#  7. Fix malformed dates (IRON_ORE has 6 rows with 'YYYY-MM' missing day)
# Reconstruct from Year and Month columns where Date is not YYYY-MM-DD
malformed = ~df['Date'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$')
print(f"\n[7a] Malformed dates found: {malformed.sum()} rows")
print(df.loc[malformed, ['Date', 'Corridor', 'Year', 'Month']])
df.loc[malformed, 'Date'] = (
    df.loc[malformed, 'Year'].astype(str) + '-' +
    df.loc[malformed, 'Month'].astype(str).str.zfill(2) + '-01'
)
print("     Reconstructed from Year/Month columns.")


[7a] Malformed dates found: 6 rows
        Date  Corridor  Year  Month
135  2016-04  IRON_ORE  2016      4
136  2016-05  IRON_ORE  2016      5
137  2016-06  IRON_ORE  2016      6
152  2017-09  IRON_ORE  2017      9
153  2017-10  IRON_ORE  2017     10
154  2017-11  IRON_ORE  2017     11
     Reconstructed from Year/Month columns.


In [16]:
#  Convert Date to datetime and sort 
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Corridor', 'Date']).reset_index(drop=True)
print(f"\n[7] Date converted to datetime. Sorted by Corridor, Date.")


[7] Date converted to datetime. Sorted by Corridor, Date.


In [17]:
#  8. Linear interpolation within each corridor 
numeric_cols = [
    'Rail_Vol_mt', 'Road_Vol_Est_mt', 'Avg_Delay_Hrs', 'Loco_Avail_Rate',
    'Security_Incidents', 'Rainfall_mm'
]
print(f"\n[8] Interpolating NaNs (linear, within corridor):")
print(f"    NaN count before: {df[numeric_cols].isnull().sum().sum()}")
 
df[numeric_cols] = (
    df.groupby('Corridor')[numeric_cols]
    .transform(lambda s: s.interpolate(method='linear', limit_direction='both'))
)
 
print(f"    NaN count after:  {df[numeric_cols].isnull().sum().sum()}")


[8] Interpolating NaNs (linear, within corridor):
    NaN count before: 21
    NaN count after:  0


In [18]:
print(f"\n── Final dataset ──────────────────────────────────────────")
print(f"Shape: {df.shape}")
print(f"Corridors: {sorted(df['Corridor'].unique())}")
print(f"Rows per corridor:\n{df['Corridor'].value_counts().sort_index()}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Total NaN: {df.isnull().sum().sum()}")
print(f"\nDescribe:\n{df.describe().round(3)}")


── Final dataset ──────────────────────────────────────────
Shape: (720, 14)
Corridors: ['CAPE', 'IRON_ORE', 'NATCOR', 'NE_CORRIDOR', 'NORTHCOR', 'SOUTHCOR']
Rows per corridor:
Corridor
CAPE           120
IRON_ORE       120
NATCOR         120
NE_CORRIDOR    120
NORTHCOR       120
SOUTHCOR       120
Name: count, dtype: int64
Date range: 2015-01-01 to 2024-12-01
Total NaN: 0

Describe:
                      Date      Year    Month  Rail_Vol_mt  Road_Vol_Est_mt  \
count                  720   720.000  720.000      720.000          720.000   
mean   2019-12-16 10:48:00  2019.499    6.500        2.215            2.801   
min    2015-01-01 00:00:00  2015.000    1.000        0.365            0.216   
25%    2017-06-23 12:00:00  2017.000    3.750        0.421            0.770   
50%    2019-12-16 12:00:00  2019.500    6.500        0.972            1.150   
75%    2022-06-08 12:00:00  2022.000    9.250        5.014            3.652   
max    2024-12-01 00:00:00  2024.000   12.000        6.269 

In [19]:
out_path = './data/corridor_panel_clean.csv'
df.to_csv(out_path, index=False)
print(f"\n✓ Saved to: {out_path}")


✓ Saved to: ./data/corridor_panel_clean.csv
